## The Stigler Diet

### Part (a) 

In [9]:
#You will probably need to run "Pkg.add(...)" before using these packages
using DataFrames, CSV, NamedArrays 

#Load the data file (ref: Boyd/263)
raw = CSV.read("stigler.csv");

# turn DataFrame into array
diet_array = convert(Array,raw); 

# the names of the DataFrame (header) are nutrients
nutrients = names(raw[2:end]);

# create a list of foods from the diet array
foods = diet_array[2:end,1];

# create a list of the minimum requirements of each nutrient
min_req = Dict(zip(nutrients,diet_array[1,2:end]));

using NamedArrays
recipe_matrix = diet_array[2:end,2:end]
recipe = NamedArray(recipe_matrix, (foods, nutrients), ("foods","nutrients"))

using JuMP, Clp

m = Model(solver=ClpSolver())

# create variables representing amount of each food to eat each day
@variable(m, x[foods] >= 0) 

# objective is to minimize the cost/day of meeting diet requirements
@objective(m, Min, sum(x[t] for t in foods))

# for each nutrient, we must eat at least the recommended daily amount
@constraint(m, constr[i in nutrients], sum(recipe[t,i] * x[t] for t in foods) >= min_req[i] );

solve(m)

println("Annual cost of this diet \$", round(365.25*getobjectivevalue(m),2))
for f in foods
    if getvalue(x[f]) > 0.0001
        println("Eat ", round(365.25*getvalue(x[f]),2), " \$1 units of ", f, " per year")
    end
end



Annual cost of this diet $39.69
Eat 10.78 $1 units of Wheat Flour (Enriched) per year
Eat 0.69 $1 units of Liver (Beef) per year
Eat 4.1 $1 units of Cabbage per year
Eat 1.83 $1 units of Spinach per year
Eat 22.29 $1 units of Navy Beans, Dried per year


Our diet is cheaper than Stigler's by 24 cents (Stigler got pretty close!). However, our diet will be pretty boring with a lot of wheat flour and navy beans and some cabbage, spinach, and beef liver.

### Part (b)

Your answers for this part will vary! Your answer will only change from part (a) if you delete at least one of the foods from the optimal diet. For example, if I delete the foods Hominy Grits, Oleomargarine, Crisco, Lard, Plate, Liver, Salt Pork, Veal Cutlets, Pork and Beans, Lima Beans, and Navy Beans, my diet becomes:

Annual cost of this diet \$42.05

Eat 21.69 \$1 units of Wheat Flour (Enriched) per year

Eat 5.57 \$1 units of Evaporated Milk (can) per year

Eat 9.0 \$1 units of Cheese (Cheddar) per year

Eat 4.26 \$1 units of Cabbage per year

Eat 1.52 \$1 units of Spinach per year

It's a bit more expensive (about \$3 more), but at least I can eat cheese now.

## Standard Form

Original Model:

$$ \begin{align}
\underset{x_1,x_2,x_3}{\min} \ & |x_1| - 4x_3\\
\text{subject to } \ & x_1 - 2x_2 = x_3\\
& x_2 \geq -x_3\\
& x_1 \text{ free}\\
& 2 \leq x_2 \leq 4\\
& x_3 \leq 0
\end{align}$$

Before we can code this model in JuMP, we have to turn it into a linear program. We can use the epigraph trick on the absolute value (since we're minimizing a convex function) and we get:

$$ \begin{align}
\underset{t,x_1,x_2,x_3}{\min} \ & t - 4x_3\\
\text{subject to } \ & x_1 - 2x_2 = x_3\\
& x_2 \geq -x_3\\
& t \geq x_1, t \geq -x_1\\
& x_1 \text{ free}\\
& 2 \leq x_2 \leq 4\\
& x_3 \leq 0
\end{align}$$

In [20]:
using JuMP, Clp

m = Model(solver=ClpSolver())

@variable(m, t >= 0)
@variable(m, x1)
@variable(m, 2 <= x2 <= 4)
@variable(m, x3 <= 0)

@objective(m, Min, t - 4*x3)

@constraint(m, x1 - 2x2 == x3)
@constraint(m, x2 >= -x3)
@constraint(m, t >= x1)
@constraint(m, t >= -x1)

solve(m)

println("Objective: ",getobjectivevalue(m))
println("x1 = ", getvalue(x1))
println("x2 = ", getvalue(x2))
println("x3 = ", getvalue(x3) )


Objective: 4.0
x1 = 4.0
x2 = 2.0
x3 = 0.0


In standard form, we must have a maximization problem with $$Ax \leq b$$ and $$x \geq 0$$. The standard form model is:

$$ \begin{align}
-\underset{t,u_1,v_1,w_2,w_3}{\max} \ & -t - 4w_3\\
\text{subject to } \ & u_1 - v_1 - 2w_2 + w_3 \leq 4\\
& -u_1 + v_1 + 2w_2 - w_3 \leq -4\\
& -w_2 + w_3 \leq 2\\
& -t + u_1 - v_1 \leq 0\\
& -t - u_1 + v_1 \leq 0\\
& w_2 \leq 2\\
& t, u_1, v_1, w_2, w_3 \geq 0,
\end{align}$$

where $$x_1 = u_1 - v_1$$ $$x_2 = w_2 + 2$$ $$x_3 = -w_3.$$

We have $$x = \begin{bmatrix}t \\ u_1 \\ v_1 \\ w_2 \\ w_3 \end{bmatrix}, \ c = \begin{bmatrix}-1\\0\\0\\0\\-4\end{bmatrix}, \ b = \begin{bmatrix}4\\-4\\2\\0\\0\\0\end{bmatrix}, \ A = \begin{bmatrix}0 & 1 & -1 & -2 & 1 \\ 0 & -1 & 1 & 2 & -1\\ 0 & 0 & 0 & -1 & 1\\ -1  & 1 & -1 & 0 & 0\\ -1 & -1 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 & 0\end{bmatrix}. $$

We can check that the answer is the same:

In [1]:
using JuMP, Clp

m = Model(solver=ClpSolver())

@variable(m,t >= 0)
@variable(m,u1 >= 0)
@variable(m,v1 >= 0)
@variable(m,w2 >= 0)
@variable(m,w3 >= 0)

@objective(m, Max, -t - 4w3)

@constraint(m, u1 - v1 - 2w2 + w3 <= 4)
@constraint(m, -u1 + v1 + 2w2 - w3 <= -4)
@constraint(m, -w2 + w3 <= 2)
@constraint(m, -t + u1 - v1 <= 0)
@constraint(m, -t - u1 + v1 <= 0)
@constraint(m, w3 <= 2)

solve(m)

println("Original objective: ", -getobjectivevalue(m))
println("x1 = ", getvalue(u1) - getvalue(v1))
println("x2 = ", getvalue(w2) + 2)
println("x3 = ", -getvalue(w3))



Original objective: 4.0
x1 = 4.0
x2 = 2.0
x3 = -0.0


## Mars Needs Noms!

There are a couple equivalent ways to model this problem (and a couple of elements in the problem statement that could be interpreted multiple ways), so your answers may differ from mine. Make sure you have the same key components, though, such as inventory balance, production rate balance, inventory limits, and some measure of shortage.

In [36]:
using JuMP, Clp

months = [1 2 3 4 5 6]
demand = Dict(zip(months, [500 800 1900 1200 700 100]))

init_inv = 200 # starting inventory (tons)
init_prod = 600 # month 0 production rate (tons)

m = Model(solver=ClpSolver())

@variable(m, x[months] >= 0) # monthly production rate (tons)
@variable(m, u[months] >= 0) # monthly production rate increase (tons)
@variable(m, ℓ[months] >= 0) # monthly production rate decrease (tons)
@variable(m, L_under[months] >= 0) # monthly leftover tons below the 800 ton limit
@variable(m, L_over[months] >= 0) # monthly leftover tons above the 800 ton limit
@variable(m, I[months]) # monthly inventory (leftover - shortage) -- can be negative
# note you could require I[6] >= 0 to ensure all demand is met by the end of month 6
@variable(m, S[months] >= 0) # monthly shortage (tons)

# minimize the sum of all the costs. you might include month 0 costs here (I do not)
@objective(m, Min, 1.5*sum(u) + sum(ℓ) + 0.4*sum(L_under) + 0.6*sum(L_over) + 2*sum(S))

# inventory identity for every month
@constraint(m, inv_identity[t in months], I[t] == L_under[t] + L_over[t] - S[t])

# monthly balance constraints (months 2-6)
@constraint(m, balance[t in months[2:6]], x[t] + I[t-1] == demand[t] + I[t])
# balance cosntraint for month 1
@constraint(m, balance_initial, x[1] + init_inv == demand[1] + I[1])

# monthly production rate calculation (months 2-6)
@constraint(m, prod_rate[t in months[2:6]], x[t] == x[t-1] + u[t] - ℓ[t])
# production rate identiy in month 1
@constraint(m, prod_initial, x[1] == init_prod + u[1] - ℓ[1])

# max inventory in each month
@constraint(m, inv_max[t in months], L_under[t] <= 800)

solve(m)

println("Total cost: ", getobjectivevalue(m))
println("Production schedule: ", getvalue(x))
println("Shortage each month: ", getvalue(S))
println("Existing inventory each month: ", getvalue(L_under))
println("Additional inventory purchased each month: ", getvalue(L_over))


Total cost: 2035.0
Production schedule: x: 1 dimensions:
[1] = 749.9999999999999
[2] = 1150.0
[3] = 1150.0
[4] = 1150.0
[5] = 700.0
[6] = 700.0
Shortage each month: S: 1 dimensions:
[1] = 0.0
[2] = 0.0
[3] = 0.0
[4] = 0.0
[5] = 0.0
[6] = 0.0
Existing inventory each month: L_under: 1 dimensions:
[1] = 449.9999999999999
[2] = 800.0
[3] = 50.0
[4] = 0.0
[5] = 0.0
[6] = 600.0
Additional inventory purchased each month: L_over: 1 dimensions:
[1] = 0.0
[2] = 0.0
[3] = 0.0
[4] = 0.0
[5] = 0.0
[6] = 0.0


We are able to meet all demand for a total cost of \$2035. At the end of each month, we have 450, 800, 50, 0, 0, and 600 tons of ice in storage. Our production rate increases to 750 tons in month 1, to 1150 tons in month 2, and decreases to 700 tons in month 5. 